# ExaHyPE 2 Tutorials - Part 3: The Euler Equations over an Airfoil

In this tutorial we change the numerical scheme from ADER-DG to
a **Finite Volumes (FV)** solver - a shock-capturing scheme that
copes well with the strong gradients of compressible flow. The configuration and
implementation are provided; read through them and then experiment.

We simulate air flowing past a symmetric **NACA&nbsp;0012** airfoil and watch a
*bow shock* form in front of it.

## The Euler Equations

The compressible Euler equations conserve density $\rho$, momentum
$\rho\mathbf{u}$ and total energy $E$:

$$
\frac{\partial}{\partial t}
\begin{pmatrix} \rho \\ \rho u_1 \\ \rho u_2 \\ E \end{pmatrix}
+ \nabla \cdot
\begin{pmatrix}
 \rho u_1 & \rho u_2 \\
 \rho u_1^2 + p & \rho u_1 u_2 \\
 \rho u_2 u_1 & \rho u_2^2 + p \\
 (E + p)u_1 & (E + p)u_2
\end{pmatrix} = \mathbf{0}.
$$

For an ideal gas the pressure is
$p = (\gamma - 1)\big(E - \tfrac{1}{2}\rho(u_1^2 + u_2^2)\big)$ with
$\gamma = 1.4$ (dry air), and the sound speed is $c = \sqrt{\gamma p / \rho}$.
The eigenvalues in direction $n$ are $u_n - c,\ u_n,\ u_n + c$.

In code the conserved state is stored as density, the two momenta, and the energy
- **not** the velocities - so remember to divide by $\rho$ where the
formulae use $u$.

## Importing the API

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../api/"))
import exahype2

## Defining the Project

The airfoil sits in a wide $120 \times 120$ channel. The offset places the
leading edge near the inflow on the left.

Where the mesh is refined is chosen at **run time**, not here: we will pass
`-lmin 2 -lmax 4` to the executable, so the whole domain is covered by a coarse
level-2 grid and ExaHyPE is *allowed* to refine down to level 4 wherever our
solver asks for it (next section). Each refinement splits a cell into
$k \times k$ cells, with k the subdivision factor of the active curve (3 on the
Peano curve, 2 on the Hilbert curve). Level 4 is $k^2$ times finer per direction than level 2.
Like the refinement in Part 2, the refined region here is **static** -
a fixed band around the airfoil that is built once and never moves.

In [ ]:
my_project = exahype2.Project(
    namespace    = ["tutorials", "exahype2", "euler"],
    project_name = "Airfoil",
    executable   = "Airfoil",
    dimensions   = 2,
    size         = [120.0, 120.0],
    offset       = [-10.0, -60.0],
    periodic_bc  = [False, False],
)

## Defining the Solver

This time we create a **Finite Volumes** solver (Godunov-type) named `FVSolver`.
The parameter `patch_size` controls how many finite volumes are grouped into one
patch that is updated together: with `patch_size = 8` each patch is an
$8 \times 8$ block of cells.

The conserved unknowns are density `rho`, the two momenta `j`, and total energy
`E`.

To resolve the airfoil without paying for a fine grid *everywhere*, we hand the
solver a **refinement criterion**. We set it to `True`,
so we write it ourselves in C++ (`refinementCriterion` in `FVSolver.cpp`): it
returns `Refine` for patches that sit near the airfoil and `Keep` otherwise.
Together with the `-lmin 2 -lmax 4` levels we pass at run time, this produces a
*static* refined band that hugs the airfoil while the far field stays coarse.

In [ ]:
fv_solver = exahype2.fv.Solver(
    name                 = "FVSolver",
    patch_size           = 8,
    unknowns             = {"rho": 1, "j": 2, "E": 1},
    auxiliary_variables  = {},
    time_stepping        = exahype2.TimeStepping.Adaptive(0.5),
    scheme               = exahype2.fv.Godunov(),
)

fv_solver.set_pde_terms(
    initial_condition    = True,
    boundary_conditions  = True,
    refinement_criterion = True,
    flux                 = True,
    max_eigenvalue       = True,
)

my_project.add_solver(fv_solver)

## Building the Project

In [ ]:
my_project.output.makefile.add_cpp_file("FVSolver.cpp")
my_project.generate_and_build()

## The Implementation File

Open **`FVSolver.cpp`**. It is complete; the interesting parts are:

* `flux` and `maxEigenvalue` - the Euler flux and $|u_n| + c$. The conserved
  variables $(\rho, \rho u_1, \rho u_2, E)$ are addressed through the generated
  `Shortcuts` enum (`Shortcuts::rho`, `Shortcuts::j + 0/1`, `Shortcuts::E`).
* `boundaryConditions` - a fixed inflow state on the left face and
  *outflow* (copy the inside state) elsewhere.
* `initialCondition` - a dense gas blob (`rho = 1000`) inside the airfoil
  contour against a uniform low-density free stream (`rho = 1`). The airfoil is
  the symmetric NACA&nbsp;0012 profile $y_t(x)$ from `airfoilSymmetric`.
* `refinementCriterion` - the **static AMR** rule. It looks only at a
  patch's position (never at `Q`, which is not valid during grid construction)
  and returns `Refine` for patches within `RefinementMargin` of the NACA
  contour, so the refined band is fixed for the whole run.

This is a deliberately simplified setup that produces a recognisable bow-shock
picture; it is not a validated aerodynamics benchmark.

## Running the Simulation

Finite Volumes are explicit, so this airfoil run is the heaviest of these
tutorials. We pass `-lmin 2 -lmax 4`: the coarsest (background) level is 2 and
the solver refines to level 4 around the airfoil, exactly where the criterion
asks for it. Watch the `Grid` and `Load` lines in the log - because the
refinement is static, the grid is built once and then time stepping begins.

In [ ]:
import os, subprocess

# Runtime parameters - edit these freely, no rebuild needed.
end_time  = 10.0    # -et   : physical end time
snapshots = 20      # -ns   : number of output snapshots (drives the animation)
min_level = 2       # -lmin : coarse background mesh level
max_level = 4       # -lmax : finest mesh level around the airfoil

# Run with 4 OpenMP threads (the notebook equivalent of: export OMP_NUM_THREADS=4)
os.environ["OMP_NUM_THREADS"] = "4"

subprocess.run(
    ["./Airfoil", "-et", str(end_time), "-ns", str(snapshots), "-lmin", str(min_level), "-lmax", str(max_level)],
    check=True,
)

## Visualising the Result

Plot the density `rho` and step through time. A curved bow shock builds up in
front of the airfoil where the incoming flow is compressed.

In [ ]:
import importlib, os, sys
sys.path.insert(0, os.path.abspath("../../"))
from postprocessing import render_solution
importlib.reload(render_solution)
render_solution.show_solution("FVSolver", field="rho", clim=(1, 2))

## Experiment

In the viewer above, use the field dropdown to show the `RefinementLevel` (the static refined band hugging the airfoil against the coarse far field) or a different physical field such as momentum `j` or energy `E` - no rebuild needed.

For the other experiments (changing `RefinementMargin`, the inflow state, or the refinement levels) see the documentation page, *Part 3 - The Euler Equations over an Airfoil*. In **Part 4 - A 2D Euler Shock with a-posteriori Limiting** you will combine both schemes: an a-posteriori limiter that runs ADER-DG everywhere but falls back to Finite Volumes in the troubled cells around a shock.